<a href="https://colab.research.google.com/github/RaniaChaudhry511/AI-ML-learning-assistant/blob/main/AI_ML_Learning_Assistant_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI & ML Learning Assistant

This project is a domain-specific AI chatbot designed to help users learn and understand concepts related to Artificial Intelligence and Machine Learning.

The chatbot is built using the **Qwen2.5-0.5B-Instruct Transformer model** from Hugging Face, along with **PyTorch** and **Gradio**. It can answer questions about Artificial Intelligence, Machine Learning, Deep Learning, NLP, Generative AI, Transformers, and basic Python for AI/ML.

The goal of this project is to demonstrate how a pre-trained **Transformer-based language model** can be integrated into an interactive chatbot and guided toward a specific learning domain through system instructions.

The chatbot provides simple explanations, examples, conceptual comparisons, and basic coding assistance while maintaining the conversation history for a more natural learning experience.

### Technologies Used

* Python
* PyTorch
* Hugging Face Transformers
* Qwen2.5-0.5B-Instruct
* Gradio


In [1]:
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
# ==========================================
# 1. MODEL
# ==========================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [3]:
# ==========================================
# 2. TOKENIZER
# ==========================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [4]:
# ==========================================
# 3. TRANSFORMER MODEL
# ==========================================

print("Loading Transformer model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading Transformer model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [8]:
# ==========================================
# 4. CHATBOT FUNCTION
# ==========================================

def chatbot(message, history):

    # ------------------------------
    # System instructions
    # ------------------------------

    messages = [
        {
            "role": "system",
            "content": """
You are an AI and ML Learning Assistant.

Your main purpose is to help users learn about:
Artificial Intelligence, Machine Learning, Deep Learning,
NLP, Generative AI, Transformers, and Python for AI/ML.

Explain concepts clearly in simple and beginner-friendly language.
Break difficult concepts into step-by-step explanations.
Give examples when useful.
Provide basic Python code when requested.

Stay focused on AI and Machine Learning topics
and help users understand concepts rather than simply giving answers.
"""
        }
    ]

    # ------------------------------
    # Add previous conversation
    # ------------------------------

    for chat in history:

        if chat["role"] in ["user", "assistant"]:

            messages.append({
                "role": chat["role"],
                "content": chat["content"]
            })

    # ------------------------------
    # Add current user message
    # ------------------------------

    messages.append({
        "role": "user",
        "content": message
    })

    # ==========================================
    # PHASE 1
    # CHAT TEMPLATE / TOKENIZATION PREPARATION
    # ==========================================

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # ==========================================
    # PHASE 2
    # TEXT → TOKEN IDs
    # ==========================================

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    # ==========================================
    # PHASE 3 + PHASE 4
    # TRANSFORMER + NEXT TOKEN PREDICTION
    # ==========================================

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    # ==========================================
    # GET ONLY NEW TOKENS
    # ==========================================

    new_tokens = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]

    # ==========================================
    # PHASE 5
    # TOKEN IDs → HUMAN TEXT
    # ==========================================

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return response

In [11]:
# ==========================================
# 5. GRADIO USER INTERFACE
# ==========================================

demo = gr.ChatInterface(

    fn=chatbot,

    title=" AI & ML Learning Assistant",

    description="""
    An interactive learning assistant powered by a
    Transformer model. It helps learners understand
    AI, Machine Learning, Deep Learning, NLP,
    Generative AI, and Transformer concepts
    through simple explanations and examples.
    """,

    examples=[
        "Explain NLP in simple words.",
        "What is supervised learning?",
        "Explain how a Transformer works.",
        "What is the difference between CNN and RNN?",
        "What is overfitting and how can we prevent it?",
        "Explain Generative AI with an example."
    ],

    save_history=True
)

In [12]:
# ==========================================
# 6. START APPLICATION
# ==========================================

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74c46fd7105b81c430.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
